# 🗄️ BUỔI 3 & 4: THIẾT LẬP KẾT NỐI POSTGRESQL VÀ XÁC THỰC DỮ LIỆU BAN ĐẦU (DATA PIPELINE)

**Họ và tên:** Nguyễn Thanh Hoài  
**Nhóm thực hiện:** PolySmart  
**Dự án:** Trích xuất, Làm sạch & Phân tích Cảm nhận Người dùng Apps (Google Play Store)  
**Mục tiêu Notebook:** 
1. Kết nối đến cơ sở dữ liệu PostgreSQL cục bộ bằng thư viện `psycopg2`.
2. Kiểm tra trạng thái kết nối và danh sách các bảng/view sẵn có.
3. Trích xuất dữ liệu từ View tổng hợp (`view_app_sentiment_summary`) sang Pandas DataFrame.
4. Đánh giá sơ bộ cấu trúc dữ liệu ban đầu (Data Profiling).

## 1. Khai báo Thư viện & Thiết lập Kết nối Database (`psycopg2`)
Khởi tạo kết nối trực tiếp tới CSDL PostgreSQL `Du_an_1_valuevoice` thông qua cổng mặc định `5432`.

In [3]:
import pandas as pd
import psycopg2

# 1. Cấu hình thông số kết nối PostgreSQL
db_config = {
    "host": "localhost",
    "database": "Du_an_1_valuevoice",
    "user": "postgres",
    "password": "hoai123",  # Mật khẩu pgAdmin của bạn
    "port": "5432"
}

try:
    # 2. Mở kết nối
    conn = psycopg2.connect(**db_config)
    print("✅ Kết nối thành công đến CSDL PostgreSQL:", db_config["database"])
except Exception as e:
    print(" Lỗi kết nối CSDL:", e)

✅ Kết nối thành công đến CSDL PostgreSQL: Du_an_1_valuevoice


## 2. Kiểm tra Danh sách Bảng & View trong Database
Xác thực sự tồn tại của các bảng dữ liệu gốc (`apps`, `user_reviews`) và View tổng hợp trước khi thực hiện truy vấn.

In [4]:
# Query danh sách các bảng và views trong schema public
tables_query = """
SELECT table_name, table_type 
FROM information_schema.tables 
WHERE table_schema = 'public';
"""

df_tables = pd.read_sql(tables_query, conn)
display(df_tables)

C:\Users\Nguyen Thanh Hoai\AppData\Local\Temp\ipykernel_30508\1685106717.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(tables_query, conn)


,table_name,table_type
0,apps,BASE TABLE
1,user_reviews,BASE TABLE
2,view_app_sentiment_summary,VIEW


## 3. Truy vấn Dữ liệu từ View Tổng hợp (`view_app_sentiment_summary`)
Nạp toàn bộ dữ liệu đã được nối (JOIN) giữa bảng thông tin ứng dụng và dữ liệu cảm nhận người dùng vào Pandas DataFrame.

In [5]:
# Truy vấn View tổng hợp
query = "SELECT * FROM view_app_sentiment_summary;"
df = pd.read_sql(query, conn)

print("================ XÁC THỰC TỔNG QUAN DỮ LIỆU ================")
print(f"👉 Kích thước dữ liệu: {df.shape[0]} dòng và {df.shape[1]} cột.\n")

# Hiển thị 5 dòng đầu tiên
display(df.head())

================ XÁC THỰC TỔNG QUAN DỮ LIỆU ================
👉 Kích thước dữ liệu: 9657 dòng và 9 cột.



C:\Users\Nguyen Thanh Hoai\AppData\Local\Temp\ipykernel_30508\2018365859.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,app_id,app_name,category,rating,reviews,installs,total_reviews_analyzed,positive_reviews,negative_reviews
0,6114,Last Day on Earth: Survival,GAME,4.5,2311785,"""10",0,0,0
1,4790,Fingerprint Lock Screen Prank,TOOLS,4.1,10786,"""1",0,0,0
2,273,"""No. Color - Color by Number","Number Coloring""",NaN,0,6.9M,0,0,0
3,3936,DY TECHNICAL GYAN,BUSINESS,4.0,2,10,0,0,0
4,5761,ibis Paint X,ART_AND_DESIGN,4.6,224399,"""10",0,0,0


## 4. Kiểm tra Cấu trúc Dữ liệu Ban đầu (Data Profiling)
Đánh giá các kiểu dữ liệu, các giá trị bị khuyết thiếu (Null) và đóng kết nối an toàn.

In [6]:
# Kiểm tra chi tiết cấu trúc các cột
print("--- THÔNG TIN CHI TIẾT CÁC CỘT (DF.INFO) ---")
df.info()

# Đóng kết nối PostgreSQL sau khi trích xuất xong
conn.close()
print("\n✅ Đã đóng kết nối PostgreSQL an toàn!")

--- THÔNG TIN CHI TIẾT CÁC CỘT (DF.INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 9657 entries, 0 to 9656
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   app_id                  9657 non-null   int64  
 1   app_name                9657 non-null   str    
 2   category                9657 non-null   str    
 3   rating                  7775 non-null   float64
 4   reviews                 9657 non-null   int64  
 5   installs                9657 non-null   str    
 6   total_reviews_analyzed  9657 non-null   int64  
 7   positive_reviews        9657 non-null   int64  
 8   negative_reviews        9657 non-null   int64  
dtypes: float64(1), int64(5), str(3)
memory usage: 679.1 KB

✅ Đã đóng kết nối PostgreSQL an toàn!


## 5. Kết luận Tiến độ Buổi 3 & 4
* [x] Đã kết nối thành công tới PostgreSQL pipeline.
* [x] Dữ liệu từ View tổng hợp `view_app_sentiment_summary` trích xuất đầy đủ và đúng định dạng.
* [x] Sẵn sàng dữ liệu cho các bước xử lý làm sạch nâng cao ở **`03_data_cleaning.ipynb`**.